# Phase 1: Projector Alignment for Multimodal Lily (v5 Optimized)

This notebook trains a 2-layer MLP projector to map vision features from **SigLIP-2** (`google/siglip2-so400m-patch14-384`) into the embedding space of **Lily** (`abhinav0231/Lily-1.5b-v0.3`).

**Upgrades in v5**:
1. **Preprocessed Dataset**: Loads pre-tokenized alignment dataset from HF Hub (`abhinav0231/lily-pretrain-alignment-dataset`), eliminating CPU image decoding and streaming network latencies.
2. **Correct Layer Matching**: Uses the second-to-last layer (`hidden_states[-2]`) of SigLIP-2, resolving the architecture layer mismatch between pretraining and SFT/evaluation.
3. **Learning Rate Scheduler Fix**: Wrap the scheduler step in `accelerator.sync_gradients` to prevent the scheduler from decaying 2x too fast.
4. **tqdm speed monitoring**: Prints epoch/step metrics, running loss, training throughput (samples/sec), and ETC.
5. **Hugging Face Private Checkpointing & Auto-Resume**: Saves checkpoints (projector weights, config, optimizer/scheduler states) every 500 steps and uploads to a private HF repo. On start, automatically checks for the latest checkpoint, downloads, and resumes training seamlessly.

In [1]:
# Install dependencies
%uv pip install -q transformers accelerate datasets wandb pillow torchvision huggingface_hub tqdm

Note: you may need to restart the kernel to use updated packages.


In [1]:
# ==============================================================================
# Cell 2 — Hardware & Authentication Setup (Hugging Face & Weights & Biases)
# ==============================================================================
import os
import json
import random
import torch
import torch.nn as nn
import time
from PIL import Image
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoProcessor, AutoTokenizer, AutoModel, AutoModelForCausalLM, get_cosine_schedule_with_warmup
from accelerate import Accelerator
from huggingface_hub import login, HfApi, upload_folder, snapshot_download, get_token
import wandb

# Retrieve HF Token
HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN", "")
if not HF_TOKEN or HF_TOKEN == "YOUR_HF_TOKEN_HERE":
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN") or userdata.get("HUGGINGFACE_TOKEN") or ""
    except Exception:
        pass

if not HF_TOKEN or HF_TOKEN == "YOUR_HF_TOKEN_HERE":
    cached_token = get_token()
    if cached_token:
        HF_TOKEN = cached_token

if HF_TOKEN and HF_TOKEN != "YOUR_HF_TOKEN_HERE":
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    try:
        login(token=HF_TOKEN, add_to_git_credential=False)
        print("✅ Authenticated with Hugging Face")
    except Exception as e:
        print(f"⚠️ Hugging Face authentication note: {e}")

# WandB Authentication
WANDB_API_KEY = os.environ.get("WANDB_API_KEY", "")
if WANDB_API_KEY and WANDB_API_KEY != "YOUR_WANDB_KEY_HERE":
    wandb.login(key=WANDB_API_KEY, relogin=True)
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY
    print("✅ Authenticated with Weights & Biases")
else:
    print("ℹ️ Operating without WandB tracking.")
    os.environ["WANDB_DISABLED"] = "true"


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: abhinav0231 (abhinav0231-krmangalam) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


>> Authenticated successfully


In [2]:
# --- Model and Dataset Config ---
LLM_MODEL_ID     = "abhinav0231/Lily-1.5b-v0.3"
VISION_MODEL_ID  = "google/siglip2-so400m-patch14-384"

PREPROCESSED_DATASET_ID = "abhinav0231/lily-pretrain-alignment-dataset"

# Training Optimization Hyperparameters
BATCH_SIZE           = 16
GRAD_ACCUM_STEPS     = 3          # Effective global batch size = 48
LEARNING_RATE        = 2e-4       
NUM_EPOCHS           = 1          # 1 Epoch is standard for visual alignment to prevent overfitting
WARMUP_RATIO         = 0.05       
WEIGHT_DECAY         = 0.05
MAX_LENGTH           = 256        

# Checkpoint & Output Repos
HF_REPO_ID           = "abhinav0231/Lily-1.5b-projector-siglip2"
CHECKPOINT_REPO      = "abhinav0231/Lily-1.5b-projector-checkpoints"
OUTPUT_DIR           = "./checkpoints"
SAVE_STEPS           = 500
RESUME_FROM_CHECKPOINT = True

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [3]:
class LilyVLM(nn.Module):
    def __init__(self, vision_model_id, llm_model_id, tokenizer):
        super().__init__()

        # Load vision tower (SigLIP-2) in bfloat16
        print(">> Loading vision tower (SigLIP-2) in bfloat16...")
        self.vision_tower = AutoModel.from_pretrained(
            vision_model_id, 
            torch_dtype=torch.bfloat16,
            output_hidden_states=True
        )
        if hasattr(self.vision_tower, "vision_model"):
            self.vision_tower = self.vision_tower.vision_model
            
        self.vision_tower.config.output_hidden_states = True
        if hasattr(self.vision_tower, "encoder"):
            self.vision_tower.encoder.config.output_hidden_states = True

        # Load language model in bfloat16
        print(">> Loading language model in bfloat16...")
        self.language_model = AutoModelForCausalLM.from_pretrained(
            llm_model_id,
            torch_dtype=torch.bfloat16,
        )
        self.language_model.resize_token_embeddings(len(tokenizer))

        # 2D pooling downsampler to compress 729 tokens to 324 tokens
        self.downsampler = nn.AdaptiveAvgPool2d((18, 18))

        # MLP Projector: Linear -> SiLU (SwiGLU matched) -> Linear
        in_dim  = self.vision_tower.config.hidden_size   # 1152 for SigLIP-2
        llm_dim = self.language_model.config.hidden_size  # 1536 for Lily
        self.projector = nn.Sequential(
            nn.Linear(in_dim, 2048),
            nn.SiLU(),
            nn.Linear(2048, llm_dim)
        )

        # Freeze vision tower and LLM; only projector trains
        for param in self.vision_tower.parameters():
            param.requires_grad = False
        for param in self.language_model.parameters():
            param.requires_grad = False

        # Enable gradient checkpointing to prevent activation memory OOM
        # self.language_model.gradient_checkpointing_enable()

    def forward(self, input_ids, attention_mask, labels, pixel_values):
        # Ensure pixel_values is in bfloat16
        pixel_values = pixel_values.to(dtype=torch.bfloat16)
        
        # Extract visual features
        with torch.no_grad():
            vision_outputs = self.vision_tower(pixel_values, output_hidden_states=True)
            image_features = vision_outputs.hidden_states[-2]  # [B, 729, 1152]

        # Perform 2D average pooling: [B, 729, 1152] -> [B, 1152, 27, 27] -> [B, 1152, 18, 18] -> [B, 324, 1152]
        B, L, D = image_features.shape
        grid_size = int(L ** 0.5)
        image_features_2d = image_features.transpose(1, 2).view(B, D, grid_size, grid_size)
        pooled_features_2d = self.downsampler(image_features_2d)
        image_features = pooled_features_2d.flatten(2).transpose(1, 2)

        # Project to LLM embedding space: [B, 324, 1536]
        image_embeddings = self.projector(image_features)  

        # Get text embeddings
        text_embeddings = self.language_model.get_input_embeddings()(input_ids)

        combined_embeddings = []
        combined_masks = []
        combined_labels = []

        batch_size = input_ids.shape[0]
        image_token_id = tokenizer.convert_tokens_to_ids("<image>")

        for i in range(batch_size):
            cur_input_ids    = input_ids[i]
            cur_attention_mask = attention_mask[i]
            cur_labels       = labels[i]
            cur_image_embeds = image_embeddings[i] 

            image_pos = (cur_input_ids == image_token_id).nonzero(as_tuple=True)[0]
            if len(image_pos) == 0:
                combined_embeddings.append(text_embeddings[i])
                combined_masks.append(cur_attention_mask)
                combined_labels.append(cur_labels)
                continue

            idx = image_pos[0].item()

            left_embeds = text_embeddings[i, :idx]
            left_mask   = cur_attention_mask[:idx]
            left_labels = cur_labels[:idx]

            right_embeds = text_embeddings[i, idx + 1:]
            right_mask   = cur_attention_mask[idx + 1:]
            right_labels = cur_labels[idx + 1:]

            visual_mask   = torch.ones(cur_image_embeds.shape[0],  dtype=cur_attention_mask.dtype, device=cur_attention_mask.device)
            visual_labels = torch.full((cur_image_embeds.shape[0],), -100, dtype=cur_labels.dtype, device=cur_labels.device)

            combined_embeddings.append(torch.cat([left_embeds, cur_image_embeds, right_embeds], dim=0))
            combined_masks.append(torch.cat([left_mask, visual_mask, right_mask], dim=0))
            combined_labels.append(torch.cat([left_labels, visual_labels, right_labels], dim=0))

        # Pad to max length in batch
        max_len = max(x.shape[0] for x in combined_embeddings)
        pad_embed = self.language_model.get_input_embeddings()(
            torch.tensor([tokenizer.pad_token_id], device=input_ids.device)
        ).squeeze(0)

        padded_embeddings, padded_masks, padded_labels = [], [], []
        for i in range(batch_size):
            diff = max_len - combined_embeddings[i].shape[0]
            if diff > 0:
                padded_embeddings.append(torch.cat([combined_embeddings[i], pad_embed.repeat(diff, 1)], dim=0))
                padded_masks.append(torch.cat([combined_masks[i], torch.zeros(diff, dtype=combined_masks[i].dtype, device=combined_masks[i].device)], dim=0))
                padded_labels.append(torch.cat([combined_labels[i], torch.full((diff,), -100, dtype=combined_labels[i].dtype, device=combined_labels[i].device)], dim=0))
            else:
                padded_embeddings.append(combined_embeddings[i])
                padded_masks.append(combined_masks[i])
                padded_labels.append(combined_labels[i])

        combined_embeddings = torch.stack(padded_embeddings, dim=0)
        combined_masks      = torch.stack(padded_masks, dim=0)
        combined_labels     = torch.stack(padded_labels, dim=0)

        outputs = self.language_model(
            inputs_embeds=combined_embeddings,
            attention_mask=combined_masks,
            labels=combined_labels,
            return_dict=True,
        )
        return outputs

In [5]:
def collate_fn(batch, pad_token_id):
    input_ids      = [torch.tensor(item["input_ids"],      dtype=torch.long)  for item in batch]
    attention_mask = [torch.tensor(item["attention_mask"], dtype=torch.long)  for item in batch]
    labels         = [torch.tensor(item["labels"],         dtype=torch.long)  for item in batch]
    pixel_values   = torch.stack([item["pixel_values"] for item in batch], dim=0)

    max_len = max(x.shape[0] for x in input_ids)
    padded_input_ids, padded_mask, padded_labels = [], [], []

    for i in range(len(batch)):
        diff = max_len - input_ids[i].shape[0]
        if diff > 0:
            padded_input_ids.append(torch.cat([input_ids[i],      torch.full((diff,), pad_token_id, dtype=torch.long)], dim=0))
            padded_mask.append(     torch.cat([attention_mask[i], torch.zeros(diff,               dtype=torch.long)],  dim=0))
            padded_labels.append(   torch.cat([labels[i],         torch.full((diff,), -100,        dtype=torch.long)], dim=0))
        else:
            padded_input_ids.append(input_ids[i])
            padded_mask.append(attention_mask[i])
            padded_labels.append(labels[i])

    return {
        "input_ids":      torch.stack(padded_input_ids, dim=0),
        "attention_mask": torch.stack(padded_mask,      dim=0),
        "labels":         torch.stack(padded_labels,    dim=0),
        "pixel_values":   pixel_values,
    }

In [4]:
# Load pre-tokenized alignment dataset from HF Hub
print(f">> Downloading preprocessed alignment dataset: {PREPROCESSED_DATASET_ID}...")
raw_dataset = load_dataset(PREPROCESSED_DATASET_ID, split="train")
print(f"Loaded dataset size: {len(raw_dataset)} samples")

>> Downloading preprocessed alignment dataset: abhinav0231/lily-pretrain-alignment-dataset...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00008.parquet:   0%|          | 0.00/461M [00:00<?, ?B/s]

data/train-00001-of-00008.parquet:   0%|          | 0.00/463M [00:00<?, ?B/s]

data/train-00002-of-00008.parquet:   0%|          | 0.00/463M [00:00<?, ?B/s]

...

data/train-00005-of-00008.parquet:   0%|          | 0.00/466M [00:00<?, ?B/s]

data/train-00006-of-00008.parquet:   0%|          | 0.00/465M [00:00<?, ?B/s]

data/train-00007-of-00008.parquet:   0%|          | 0.00/462M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/121237 [00:00<?, ? examples/s]

Loaded dataset size: 121237 samples


In [6]:
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_ID)
if "<image>" not in tokenizer.get_vocab():
    tokenizer.add_tokens(["<image>"], special_tokens=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

processor = AutoProcessor.from_pretrained(VISION_MODEL_ID, use_fast=True)

from torch.utils.data import Dataset as TorchDataset

class LilyPretrainDataset(TorchDataset):
    def __init__(self, hf_dataset, processor):
        self.dataset = hf_dataset
        self.processor = processor

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        image = item["image"]
        
        # SigLIP-2 processor on the PIL Image
        try:
            pixel_values = self.processor(images=image, return_tensors="pt")["pixel_values"].squeeze(0)
        except Exception:
            dummy = Image.new("RGB", (384, 384), (0, 0, 0))
            pixel_values = self.processor(images=dummy, return_tensors="pt")["pixel_values"].squeeze(0)
            
        return {
            "input_ids": item["input_ids"],
            "attention_mask": item["attention_mask"],
            "labels": item["labels"],
            "pixel_values": pixel_values
        }

dataset = LilyPretrainDataset(raw_dataset, processor)

dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    prefetch_factor=2,
    collate_fn=lambda b: collate_fn(b, tokenizer.pad_token_id),
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

...

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/34.4M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

In [7]:
# Checkpoint Saving & HF uploading helper
def save_and_push_checkpoint(model, optimizer, lr_scheduler, step, output_dir, repo_id, token):
    ckpt_dir = os.path.join(output_dir, f"checkpoint-{step}")
    os.makedirs(ckpt_dir, exist_ok=True)
    
    # Save projector weights
    unwrapped = accelerator.unwrap_model(model)
    torch.save(unwrapped.projector.state_dict(), os.path.join(ckpt_dir, "mm_projector.bin"))
    
    # Save projector config
    proj_config = {
        "in_dim":         unwrapped.vision_tower.config.hidden_size,
        "hidden_dim":     2048,
        "out_dim":        unwrapped.language_model.config.hidden_size,
        "activation":     "gelu",
        "vision_feature": "hidden_states[-2]",
        "vision_model_id": VISION_MODEL_ID,
        "llm_model_id":   LLM_MODEL_ID,
    }
    with open(os.path.join(ckpt_dir, "projector_config.json"), "w") as f:
        json.dump(proj_config, f, indent=2)
        
    # Save optimizer and scheduler states
    torch.save(optimizer.state_dict(), os.path.join(ckpt_dir, "optimizer.bin"))
    torch.save(lr_scheduler.state_dict(), os.path.join(ckpt_dir, "scheduler.bin"))
    
    if repo_id:
        print(f"\nStep {step}: pushing checkpoint to Hugging Face {repo_id}...")
        try:
            upload_folder(
                folder_path=ckpt_dir,
                repo_id=repo_id,
                token=token,
                path_in_repo=f"checkpoint-{step}",
                commit_message=f"Phase 1 pretraining checkpoint step {step}",
                ignore_patterns=["*.lock"],
            )
            print("Checkpoint pushed successfully!")
        except Exception as e:
            print(f"WARNING: HF push failed: {e}")

In [8]:
# Auto-Resume Checkpoint Download logic
_resume_ckpt = None
if RESUME_FROM_CHECKPOINT and CHECKPOINT_REPO:
    try:
        api = HfApi()
        api.create_repo(CHECKPOINT_REPO, token=HF_TOKEN, exist_ok=True, private=True)
        
        files = list(api.list_repo_files(CHECKPOINT_REPO, token=HF_TOKEN))
        nums = set()
        for f in files:
            seg = f.split("/")[0]
            if seg.startswith("checkpoint-") and seg.split("-")[-1].isdigit():
                nums.add(int(seg.split("-")[-1]))
        if nums:
            latest = max(nums)
            local_dir = os.path.join(OUTPUT_DIR, f"checkpoint-{latest}")
            print(f"Found checkpoint-{latest} in HF repo — downloading...")
            snapshot_download(
                repo_id=CHECKPOINT_REPO,
                allow_patterns=[f"checkpoint-{latest}/*"],
                local_dir=OUTPUT_DIR,
                token=HF_TOKEN,
            )
            _resume_ckpt = local_dir
            print(f"Checkpoint downloaded -> {local_dir}")
        else:
            print("No checkpoints found on Hugging Face — starting fresh.")
    except Exception as e:
        print(f"WARNING: Resume failed ({e}) — starting fresh.")

No checkpoints found on Hugging Face — starting fresh.


In [9]:
# Initialize accelerator for BF16 & WandB tracking
accelerator = Accelerator(mixed_precision="bf16", log_with="wandb")

# Initialize model
model = LilyVLM(VISION_MODEL_ID, LLM_MODEL_ID, tokenizer)

# Optimize only the projector parameters
projector_params = [p for p in model.projector.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(
    projector_params,
    lr=LEARNING_RATE,
    betas=(0.9, 0.95),
    weight_decay=WEIGHT_DECAY,
)

num_training_steps = (len(dataloader) // GRAD_ACCUM_STEPS) * NUM_EPOCHS
num_warmup_steps = int(num_training_steps * WARMUP_RATIO)

lr_scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps,
)

# Prepare Accelerator targets
model, optimizer, dataloader, lr_scheduler = accelerator.prepare(
    model, optimizer, dataloader, lr_scheduler
)

trainable_params_count = sum(p.numel() for p in model.parameters() if p.requires_grad)

# Load checkpoint state dicts if resuming (after prepare)
start_step = 0
global_step = 0
if _resume_ckpt:
    start_step = int(_resume_ckpt.split("-")[-1])
    global_step = start_step
    
    proj_weights_path = os.path.join(_resume_ckpt, "mm_projector.bin")
    if os.path.exists(proj_weights_path):
        unwrapped = accelerator.unwrap_model(model)
        unwrapped.projector.load_state_dict(torch.load(proj_weights_path, map_location="cpu"))
        print(f">> Projector weights loaded from checkpoint.")
        
    opt_path = os.path.join(_resume_ckpt, "optimizer.bin")
    if os.path.exists(opt_path):
        optimizer.load_state_dict(torch.load(opt_path, map_location="cpu"))
        print(f">> Optimizer states loaded from checkpoint.")
        
    sch_path = os.path.join(_resume_ckpt, "scheduler.bin")
    if os.path.exists(sch_path):
        lr_scheduler.load_state_dict(torch.load(sch_path, map_location="cpu"))
        print(f">> Scheduler states loaded from checkpoint.")
        
    print(f">> Resumed training state from global step: {global_step}")

accelerator.init_trackers(
    project_name="lily-vision-phase1-alignment-v5",
    config={
        "learning_rate": LEARNING_RATE,
        "epochs": NUM_EPOCHS,
        "batch_size": BATCH_SIZE,
        "gradient_accumulation_steps": GRAD_ACCUM_STEPS,
        "trainable_params": trainable_params_count,
        "num_training_steps": num_training_steps,
    }
)

Detected kernel version 4.19.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


>> Loading vision tower (SigLIP-2) in bfloat16...


config.json:   0%|          | 0.00/559 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

>> Loading language model in bfloat16...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/164 [00:00<?, ?B/s]

wandb: setting up run 93w46t5u
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /root/wandb/run-20260717_092648-93w46t5u
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run breezy-terrain-10
wandb: ⭐️ View project at https://wandb.ai/abhinav0231-krmangalam/lily-vision-phase1-alignment-v5
wandb: 🚀 View run at https://wandb.ai/abhinav0231-krmangalam/lily-vision-phase1-alignment-v5/runs/93w46t5u


In [10]:
model.train()
import time
from tqdm.auto import tqdm

print(">> Starting Projector Alignment (Phase 1)... ")
completed_epochs = start_step // (len(dataloader) // GRAD_ACCUM_STEPS)

try:
    for epoch in range(completed_epochs, NUM_EPOCHS):
        epoch_loss = 0.0
        print(f"\n>> Epoch {epoch + 1}/{NUM_EPOCHS}")

        pbar = tqdm(
            enumerate(dataloader),
            total=len(dataloader),
            desc=f"Epoch {epoch+1}",
            disable=not accelerator.is_main_process
        )
        
        start_time = time.time()
        accumulated_loss = 0.0

        for step, batch in pbar:
            current_global_step = epoch * (len(dataloader) // GRAD_ACCUM_STEPS) + (step // GRAD_ACCUM_STEPS)
            
            if current_global_step < start_step:
                if (step + 1) % GRAD_ACCUM_STEPS == 0:
                    # Fast-forward lr scheduler and global_step counters
                    lr_scheduler.step()
                    global_step += 1
                continue
                
            with accelerator.accumulate(model):
                outputs = model(**batch)
                loss = outputs.loss

                accelerator.backward(loss)
                optimizer.step()
                if accelerator.sync_gradients:
                    lr_scheduler.step()
                optimizer.zero_grad()

            accumulated_loss += loss.item()
            
            if accelerator.sync_gradients:
                global_step += 1
                epoch_loss += accumulated_loss / GRAD_ACCUM_STEPS
                
                # Compute samples/second throughput
                end_time = time.time()
                step_time = end_time - start_time
                samples_per_sec = (BATCH_SIZE * GRAD_ACCUM_STEPS) / step_time if step_time > 0 else 0.0
                
                pbar.set_postfix({
                    "loss": f"{(accumulated_loss / GRAD_ACCUM_STEPS):.4f}",
                    "lr": f"{lr_scheduler.get_last_lr()[0]:.2e}",
                    "speed": f"{samples_per_sec:.1f} samples/s"
                })

                accelerator.log(
                    {
                        "loss": accumulated_loss / GRAD_ACCUM_STEPS,
                        "lr": lr_scheduler.get_last_lr()[0],
                        "samples_per_sec": samples_per_sec,
                        "epoch": epoch + 1,
                    },
                    step=global_step,
                )
                
                # Save checkpoint step
                if global_step > 0 and global_step % SAVE_STEPS == 0:
                    save_and_push_checkpoint(
                        model, optimizer, lr_scheduler, global_step, OUTPUT_DIR, CHECKPOINT_REPO, HF_TOKEN
                    )
                    
                accumulated_loss = 0.0
                start_time = time.time()

        pbar.close()
        avg_epoch_loss = epoch_loss / (len(dataloader) // GRAD_ACCUM_STEPS)
        print(f">> Epoch {epoch + 1} complete | Avg loss: {avg_epoch_loss:.4f}")
        accelerator.log({"avg_epoch_loss": avg_epoch_loss}, step=global_step)

    print(">> Training complete!")

except KeyboardInterrupt:
    print(">> Training interrupted by user.")

>> Starting Projector Alignment (Phase 1)... 

>> Epoch 1/1


Epoch 1:   0%|          | 0/7578 [00:00<?, ?it/s]


Step 500: pushing checkpoint to Hugging Face abhinav0231/Lily-1.5b-projector-checkpoints...


Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

...

New Data Upload                         : |          |  0.00B /  0.00B            

  ...ts/checkpoint-7500/mm_projector.bin:   1%|          |  189kB / 22.0MB            

  ...oints/checkpoint-7500/optimizer.bin:   1%|          |  377kB / 44.1MB            

  ...oints/checkpoint-7500/scheduler.bin:   1%|          |  12.0B / 1.47kB            

Checkpoint pushed successfully!
>> Epoch 1 complete | Avg loss: 1.3550
>> Training complete!


In [11]:
# Save final projector weights and push to HuggingFace Hub
if accelerator.is_main_process:
    save_dir = "checkpoints/projector_phase1_v5"
    os.makedirs(save_dir, exist_ok=True)

    unwrapped_model = accelerator.unwrap_model(model)

    torch.save(
        unwrapped_model.projector.state_dict(),
        os.path.join(save_dir, "mm_projector_final.bin")
    )
    print(f">> Saved final projector weights to {save_dir}/mm_projector_final.bin")

    # Save config so Phase 2 can verify architecture before loading
    proj_config = {
        "in_dim":         unwrapped_model.vision_tower.config.hidden_size,
        "hidden_dim":     2048,
        "out_dim":        unwrapped_model.language_model.config.hidden_size,
        "activation":     "gelu",
        "vision_feature": "hidden_states[-2]",
        "vision_model_id": VISION_MODEL_ID,
        "llm_model_id":   LLM_MODEL_ID,
    }
    config_path = os.path.join(save_dir, "projector_config.json")
    with open(config_path, "w") as f:
        json.dump(proj_config, f, indent=2)
    print(f">> Saved projector config to {config_path}")

    # Push final folder to Hub
    api = HfApi()
    api.create_repo(repo_id=HF_REPO_ID, repo_type="model", exist_ok=True, private=False)
    api.upload_folder(
        folder_path=save_dir,
        repo_id=HF_REPO_ID,
        repo_type="model",
        commit_message="Phase 1 v5: Optimized Preprocessed mixed alignment, GELU, hidden_states[-2]",
    )
    print(f">> Pushed to https://huggingface.co/{HF_REPO_ID}")

accelerator.end_training()

>> Saved final projector weights to checkpoints/projector_phase1_v5/mm_projector_final.bin
>> Saved projector config to checkpoints/projector_phase1_v5/projector_config.json


Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  ...or_phase1_v5/mm_projector_final.bin:   3%|2         |  559kB / 22.0MB            

wandb: updating run metadata


>> Pushed to https://huggingface.co/abhinav0231/Lily-1.5b-projector-siglip2


wandb: uploading wandb-summary.json; uploading config.yaml
wandb: uploading history steps 7577-7577, summary
wandb: 
wandb: Run history:
wandb:  avg_epoch_loss ▁
wandb:           epoch ▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:            loss ▆▇██▇▆▆▅▅▅▄▃▄▅▆▅▆▅▅▅▅▅▄▄▅▅▄▃▄▄▃▂▃▃▂▃▂▁▁▂
wandb:              lr ▂▇███▅▅▅▃▂▁▁▁▁▁▃▃▄▅▅▆▇▇▇▇█████▇▆▆▅▅▃▂▂▁▁
wandb: samples_per_sec ▁▂▂▂▂▁█▂▁▁▁▂▂▂▂▂▂▂▁▂▂▂▂▂▂▂▁▂▃▁▂▁▂▂▂▃▂▁▁▂
wandb: 
wandb: Run summary:
wandb:  avg_epoch_loss 1.35498
wandb:           epoch 1
wandb:            loss 0.4021
wandb:              lr 1e-05
wandb: samples_per_sec 109.6045
wandb: 
wandb: 🚀 View run breezy-terrain-10 at: https://wandb.ai/abhinav0231-krmangalam/lily-vision-phase1-alignment-v5/runs/93w46t5u
wandb: ⭐️ View project at: https://wandb.ai/abhinav0231-krmangalam/lily-vision-phase1-alignment-v5
wandb: Synced 4 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260717_092648-93w46t5u/logs
